In [1]:
import pennylane as qp
import numpy as np

In [2]:
def multisol_oracle_matrix(combos):
    """Return the oracle matrix for a set of solutions.

    Args:
        combos (list[list[int]]): A list of secret bit strings.

    Returns:
        array[float]: The matrix representation of the oracle.
    """
    indices = [np.ravel_multi_index(combo, [2]*len(combo)) for combo in combos]

    my_array = np.identity(2**len(combos[0])) # Create the identity matrix
    for index in indices:
        my_array[index, index] = -1

    return my_array

In [3]:
n_bits = 4
dev = qp.device("default.qubit", wires=n_bits)

@qp.qnode(dev)
def multisol_hoh_circuit(combos):
    """A circuit which applies Hadamard, multi-solution oracle, then Hadamard.
    
    Args:
        combos (list[list[int]]): A list of secret bit strings.

    Returns: 
        array[float]: Probabilities for observing different outcomes.
    """

    for i in range(n_bits):
        qp.H(i)
    qp.QubitUnitary(multisol_oracle_matrix(combos), wires=range(n_bits))
    for i in range(n_bits):
        qp.H(i)

    return qp.probs(wires=range(n_bits))

In [4]:
def multisol_combo(n_bits, how_many):
    # Total possible unique bitstrings for n_bits is 2**n_bits
    all_combos = [list(map(int, format(i, f'0{n_bits}b'))) for i in range(2**n_bits)]
    
    # Randomly select 'how_many' unique combinations from the pool
    indices = np.random.choice(len(all_combos), size=how_many, replace=False)
    
    return [all_combos[i] for i in indices]

In [6]:
def deutsch_jozsa(promise_var):
    """Implement the Deutsch–Jozsa algorithm and guess the promise variable.
    
    Args:
        promise_var (int): Indicates whether the function is balanced (0) or constant (1).
        
    Returns: 
        int: A guess at the promise variable.
    """
    if promise_var == 0:
        how_many = 2**(n_bits - 1)
    else:
        how_many = np.random.choice([0, 2**n_bits]) # Choose all or nothing randomly
    combos = multisol_combo(n_bits, how_many) # Generate random combinations

    probs = multisol_hoh_circuit(combos)
    if probs[0] < 0.5: 
        return 0 
    else: 
        return 1